# GPU Inference Telemetry Validation with TFDV

**Dataset**: Alibaba PAI GPU Cluster Trace v2020 (6,500+ GPUs, NSDI'22)  
**Adapted from**: MLOps Lab 3 (C2W1_Assignment)

This notebook demonstrates:
1. Generating and visualizing dataset statistics
2. Inferring and customizing a data schema with GPU-specific constraints
3. Detecting anomalies in evaluation data (injected + real)
4. Detecting temporal drift between trace halves
5. Validating serving data against the training schema

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import tensorflow_data_validation as tfdv
import pandas as pd
import numpy as np
from tensorflow_metadata.proto.v0 import schema_pb2

print(f"TFDV version: {tfdv.__version__}")

## 1. Load the Datasets

- **Training**: First half of trace period (baseline — ~July 2020)
- **Evaluation**: Second half of trace period (drift candidate — ~August 2020)
- **Serving**: Task resource requests only (no utilization labels — simulates production requests)

In [ ]:
train_df = pd.read_csv("../data/processed/training_data.csv")
eval_df = pd.read_csv("../data/processed/eval_data.csv")
serving_df = pd.read_csv("../data/processed/serving_data.csv")

print(f"Training:  {train_df.shape}")
print(f"Eval:      {eval_df.shape}")
print(f"Serving:   {serving_df.shape}")

In [ ]:
# Explore training data
print("=== First 5 Rows ===")
display(train_df.head())
print("\n=== Column Types ===")
print(train_df.dtypes)
print("\n=== GPU Type Distribution ===")
print(train_df["gpu_type"].value_counts())
print("\n=== Summary Statistics ===")
display(train_df.describe().round(2))

## 2. Generate Training Statistics

TFDV computes descriptive statistics (mean, std, min, max, histograms, missing counts) using Apache Beam under the hood. Toggle the interactive visualization to explore distributions.

In [ ]:
train_stats = tfdv.generate_statistics_from_dataframe(train_df)
tfdv.visualize_statistics(train_stats)

## 3. Infer Schema from Training Data

The schema defines expected data types, value ranges, and domains. TFDV infers a baseline schema from training statistics.

In [ ]:
schema = tfdv.infer_schema(train_stats)
tfdv.display_schema(schema)

## 4. Customize Schema — GPU Telemetry Constraints

This is the **key modification** from Lab 3. Instead of hospital data constraints, we set hardware-specific constraints:
- GPU utilization must be 0–100% (values above indicate measurement artifacts)
- CPU utilization must be 0–100%
- GPU types restricted to known hardware in this cluster
- Critical telemetry features marked as required (no nulls)

In [ ]:
# Set gpu_type as categorical with known domain
tfdv.set_domain(schema, "gpu_type", schema_pb2.StringDomain(
    value=["T4", "P100", "V100", "V100M32", "MISC"]
))

# Set value ranges for utilization features
gpu_util_feature = tfdv.get_feature(schema, "gpu_util_per_gpu")
gpu_util_feature.float_domain.min = 0.0
gpu_util_feature.float_domain.max = 100.0

cpu_feature = tfdv.get_feature(schema, "machine_cpu")
cpu_feature.float_domain.min = 0.0
cpu_feature.float_domain.max = 100.0

cpu_usr_feature = tfdv.get_feature(schema, "machine_cpu_usr")
cpu_usr_feature.float_domain.min = 0.0
cpu_usr_feature.float_domain.max = 100.0

cpu_kernel_feature = tfdv.get_feature(schema, "machine_cpu_kernel")
cpu_kernel_feature.float_domain.min = 0.0
cpu_kernel_feature.float_domain.max = 100.0

cpu_iowait_feature = tfdv.get_feature(schema, "machine_cpu_iowait")
cpu_iowait_feature.float_domain.min = 0.0
cpu_iowait_feature.float_domain.max = 100.0

cap_gpu_feature = tfdv.get_feature(schema, "cap_gpu")
cap_gpu_feature.int_domain.min = 1
cap_gpu_feature.int_domain.max = 8

# Mark required features (no nulls allowed)
for feature_name in ["machine_cpu", "gpu_util_per_gpu", "machine_load_1", "gpu_type"]:
    feature = tfdv.get_feature(schema, feature_name)
    feature.presence.min_fraction = 1.0

print("✅ Schema customized with GPU telemetry constraints")
tfdv.display_schema(schema)

In [ ]:
# Save schema artifact
schema_path = "../schema/schema.pbtxt"
tfdv.write_schema_text(schema, schema_path)
print(f"Schema saved to {schema_path}")

## 5. Compare Training vs Evaluation Statistics

Overlay distributions from both time periods to visually inspect workload drift. Key things to watch:
- GPU type distribution shift (T4 vs P100 ratio changes over time)
- `gpu_util_per_gpu` distribution changes (heavier/lighter workloads)
- `machine_load_1` changes (cluster load patterns)

In [ ]:
eval_stats = tfdv.generate_statistics_from_dataframe(eval_df)

tfdv.visualize_statistics(
    lhs_statistics=eval_stats,
    rhs_statistics=train_stats,
    lhs_name="Eval (Second Half)",
    rhs_name="Training (First Half)"
)

## 6. Inject Synthetic Anomalies

Following the Lab 3 pattern (`add_extra_rows`), we inject known anomalies into the eval dataframe to test schema validation. These simulate real production data quality issues:

| # | Anomaly | What it simulates |
|---|---|---|
| 1 | Negative GPU utilization (-25%) | Sensor glitch / faulty reading |
| 2 | GPU utilization = 150% | Extreme measurement artifact |
| 3 | NaN in required fields | Missing telemetry data |
| 4 | Unknown GPU type "A100" | New hardware not in schema |
| 5 | CPU utilization = 113% | Impossible value |

In [ ]:
anomalous_rows = pd.DataFrame([
    {   # Anomaly 1: Negative GPU utilization (sensor glitch)
        "machine_cpu_usr": 15.0, "machine_cpu_kernel": 3.0,
        "machine_cpu_iowait": 0.0, "machine_cpu": 18.0,
        "machine_gpu": -50.0, "gpu_util_per_gpu": -25.0,
        "machine_load_1": 10.0, "machine_net_receive": 1e8,
        "machine_num_worker": 4.0, "gpu_type": "T4",
        "cap_cpu": 96, "cap_mem": 512, "cap_gpu": 2, "duration_sec": 300
    },
    {   # Anomaly 2: GPU utilization > 100%
        "machine_cpu_usr": 30.0, "machine_cpu_kernel": 5.0,
        "machine_cpu_iowait": 0.0, "machine_cpu": 35.0,
        "machine_gpu": 1200.0, "gpu_util_per_gpu": 150.0,
        "machine_load_1": 50.0, "machine_net_receive": 5e8,
        "machine_num_worker": 8.0, "gpu_type": "P100",
        "cap_cpu": 96, "cap_mem": 512, "cap_gpu": 8, "duration_sec": 600
    },
    {   # Anomaly 3: Missing critical values
        "machine_cpu_usr": 20.0, "machine_cpu_kernel": 4.0,
        "machine_cpu_iowait": 0.0, "machine_cpu": np.nan,
        "machine_gpu": 100.0, "gpu_util_per_gpu": np.nan,
        "machine_load_1": np.nan, "machine_net_receive": 2e8,
        "machine_num_worker": 5.0, "gpu_type": "V100",
        "cap_cpu": 64, "cap_mem": 384, "cap_gpu": 2, "duration_sec": 450
    },
    {   # Anomaly 4: Unknown GPU type
        "machine_cpu_usr": 25.0, "machine_cpu_kernel": 6.0,
        "machine_cpu_iowait": 0.0, "machine_cpu": 31.0,
        "machine_gpu": 200.0, "gpu_util_per_gpu": 50.0,
        "machine_load_1": 20.0, "machine_net_receive": 3e8,
        "machine_num_worker": 6.0, "gpu_type": "A100",
        "cap_cpu": 96, "cap_mem": 512, "cap_gpu": 4, "duration_sec": 500
    },
    {   # Anomaly 5: CPU utilization > 100%
        "machine_cpu_usr": 105.0, "machine_cpu_kernel": 8.0,
        "machine_cpu_iowait": 0.0, "machine_cpu": 113.0,
        "machine_gpu": 80.0, "gpu_util_per_gpu": 40.0,
        "machine_load_1": 30.0, "machine_net_receive": 2e8,
        "machine_num_worker": 4.0, "gpu_type": "T4",
        "cap_cpu": 96, "cap_mem": 512, "cap_gpu": 2, "duration_sec": 350
    },
])

eval_df_with_anomalies = pd.concat([eval_df, anomalous_rows], ignore_index=True)
print(f"Eval data: {len(eval_df)} → {len(eval_df_with_anomalies)} rows (+{len(anomalous_rows)} anomalies)")

## 7. Validate Eval Data Against Schema

TFDV checks every constraint we defined in Step 4. Expected detections:
- `gpu_util_per_gpu` out of range [0, 100] → Anomalies 1, 2 + real data artifact (207.6%)
- `machine_cpu` out of range [0, 100] → Anomaly 5
- `machine_cpu_usr` out of range → Anomaly 5
- Missing required values → Anomaly 3
- Unknown `gpu_type` "A100" → Anomaly 4

In [ ]:
eval_stats_anomalous = tfdv.generate_statistics_from_dataframe(eval_df_with_anomalies)
anomalies = tfdv.validate_statistics(eval_stats_anomalous, schema)
tfdv.display_anomalies(anomalies)

## 8. Drift Detection

Drift detection compares distributions between training and evaluation data using:
- **L-infinity distance** for numeric features
- **Jensen-Shannon divergence** for categorical features

We use the *clean* eval data (without injected anomalies) to detect **real** temporal drift.

In [ ]:
# Set drift comparators
for feat_name in ["gpu_util_per_gpu", "machine_cpu", "machine_load_1"]:
    feat = tfdv.get_feature(schema, feat_name)
    feat.drift_comparator.infinity_norm.threshold = 0.01

gpu_type_feat = tfdv.get_feature(schema, "gpu_type")
gpu_type_feat.drift_comparator.jensen_shannon_divergence.threshold = 0.01

drift_anomalies = tfdv.validate_statistics(
    eval_stats, schema, previous_statistics=train_stats
)
tfdv.display_anomalies(drift_anomalies)

## 9. Validate Serving Data

Serving data has a **completely different schema** (6 columns vs 14). It contains resource *requests* (`plan_cpu`, `plan_mem`, `plan_gpu`) but no utilization measurements. This mirrors Lab 3's serving set that lacks the label column.

TFDV should flag all missing utilization columns and all new request columns.

In [ ]:
print(f"Serving columns: {list(serving_df.columns)}")
print(f"Training columns: {list(train_df.columns)}")
print(f"\nMissing in serving: {set(train_df.columns) - set(serving_df.columns)}")

serving_stats = tfdv.generate_statistics_from_dataframe(serving_df)

tfdv.visualize_statistics(
    lhs_statistics=serving_stats,
    rhs_statistics=train_stats,
    lhs_name="Serving (Resource Requests)",
    rhs_name="Training (Utilization Metrics)"
)

In [ ]:
serving_anomalies = tfdv.validate_statistics(serving_stats, schema)
tfdv.display_anomalies(serving_anomalies)

## 10. Summary

| Validation Step | Result |
|---|---|
| Schema inferred | ✅ 14 features, types auto-detected |
| Schema customized | ✅ GPU-specific ranges, domains, required fields |
| Injected anomalies detected | ✅ All 5 caught (negative util, OOR, nulls, unknown type, CPU>100%) |
| Real data anomaly found | ✅ gpu_util_per_gpu = 207.6% (measurement artifact in Alibaba trace) |
| Temporal drift detected | ✅ Distribution shift between trace halves |
| Serving schema mismatch | ✅ 13 missing columns, 5 new columns flagged |